# Build ORR Free Energy Diagrams


## Goal of the notebook

Let us build an ORR free energy diagram from adsorption calculations. The diagram helps us find the thermodynamic limiting potential.


## Description of the method

The Computational Hydrogen Electrode links proton and electron transfers to hydrogen and electrode potential. Consistent molecular references and corrections place OOH, O, and OH on one free energy scale. The ORR step energies then show the potential at which the least favorable reduction step reaches zero free energy.

We will use two data routes: the UMA potential and literature BEEF-vdW databases. As we compare them, let us keep each source's reference energies, corrections, and assumptions in view.


## What we will do

1. Let us prepare molecular references and calculate adsorption energies with ASE.
2. We will apply the stated corrections and draw diagrams at chosen potentials.
3. We will find limiting potentials from UMA results and literature DFT data.


In [ ]:
# Install course dependencies only in Google Colab. Local runs use environment.yml.
!if python -c "import google.colab" >/dev/null 2>&1; then pip install -q ipywidgets fairchem-core ase gdown matplotlib numpy pandas; fi

### If the Colab install reports dependency conflicts

Colab may already include `torchvision` and `pytensor`. Installing FairChem can update PyTorch and Numba to versions those packages do not request. This warning alone does not mean the installation failed. These notebooks do not use `torchvision` or PyTensor, so you can leave them installed and continue without restarting the runtime. If a FairChem import or calculation shows an actual error, use that error to guide the next step.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import re
import time

from ase import Atom, Atoms
from ase.build import add_adsorbate, fcc111, graphene_nanoribbon, molecule
from ase.constraints import FixAtoms, FixedLine
from ase.data import atomic_numbers, reference_states
from ase.data.pubchem import pubchem_atoms_search
from ase.db import connect
from ase.io import read, write
from ase.io.trajectory import Trajectory
from ase.optimize import BFGS
from ase.visualize import view
from dataclasses import dataclass
from fairchem.core import FAIRChemCalculator, pretrained_mlip
from functools import partial
from IPython.display import display
from ipywidgets import Checkbox, FloatSlider, HBox, interactive_output, VBox
from pathlib import Path
from torch.serialization import add_safe_globals

In [ ]:
# Standard publication figure sizes in inches
plt.rc("figure", figsize  = (8.25 / 2.54, 8.25 / 2.54))
plt.rc("figure", dpi      = 100)
plt.rc("lines",  linewidth= 4)
plt.rc("font",   size     = 10)
plt.rc("font",   family   = "sans-serif")
plt.rc("axes",   titlesize= 10)
plt.rc("axes",   labelsize= 9)
plt.rc("xtick",  labelsize= 8)
plt.rc("ytick",  labelsize= 8)
plt.rc("legend", fontsize = 8)
plt.rc("figure", titlesize= 10)

pd.options.display.float_format = "{:.2f}".format

The UMA model is distributed under the [FAIR Chemistry License](https://huggingface.co/facebook/UMA/blob/main/LICENSE). Let us read its terms before we use the model. The full setup for your own account requires a Hugging Face access token. This notebook provides the model file through a Google Drive link, so each learner can follow the calculation here.


In [ ]:
# Download once into the current folder.
!if [ ! -s uma-s-1p1.pt ]; then gdown 1pw3jkPMWL6CLmpRGhBPdvzoLn4MzpTEX -O uma-s-1p1.pt; fi
if not Path('uma-s-1p1.pt').is_file() or Path('uma-s-1p1.pt').stat().st_size == 0:
    raise FileNotFoundError("Download failed: uma-s-1p1.pt")

## Set plot conventions

Let us set the figure size, fonts, and colors once, then use those choices in the plots below.


In [ ]:
colors = ["#1965B0", "#7BAFDE", "#90C987", "#F1932D", "#DC050C"]

## Prepare the calculator and adsorbate calculation

We will load UMA once. The function below repeats the same relaxation steps for O, OH, and OOH. A function stores its instructions; it runs when we call it. The next cells build the clean slab and show the free energy expressions.


In [ ]:
# get_potential_energy asks the attached calculator for the electronic energy in eV.
add_safe_globals([slice])  # Permit slice objects stored in the trusted UMA checkpoint.
predict_unit = pretrained_mlip.load_predict_unit("uma-s-1p1.pt", device="cpu")
calculator = FAIRChemCalculator(predict_unit, task_name="omat")

In [ ]:
# .copy() creates an independent object so changes do not overwrite the original.
def relax_adsorbate(slab, adsorbate, position, metal="Pt", fmax=0.05, maxsteps=115):
    atoms = slab.copy()
    adsorbates = {
        "O": (Atoms("O", positions=[(0, 0, 0)]), 1.5),
        "OH": (Atoms("OH", positions=[(0, 0, 0), (0.7, 0, 0.7)]), 1.8),
        "OOH": (Atoms("OOH", positions=[(0, 0, 0), (0.1, 0, 1.5), (0.7, 0, 2.2)]), 1.8),
    }
    ads, height = adsorbates[adsorbate]
    add_adsorbate(atoms, ads, height, position)
    atoms.set_constraint(FixAtoms(mask=atoms.get_tags() > 0))
    atoms.calc = calculator
    optimizer = BFGS(atoms, trajectory=f"{metal}_{adsorbate}.traj", logfile=f"{metal}_{adsorbate}.log")
    optimizer.run(fmax=fmax, steps=maxsteps)
    return atoms.get_potential_energy()

## Calculate molecular references

H₂ and H₂O appear in each Hess cycle. Let us optimize these molecules first and save their reference energies.


In [ ]:
h2 = molecule("H2")
h2.calc = calculator
optimizer = BFGS(h2, trajectory="H2.traj", logfile="H2.log")
optimizer.run(fmax=0.01)
energy_h2_ev = h2.get_potential_energy()
h2o = molecule("H2O")
h2o.calc = calculator
optimizer = BFGS(h2o, trajectory="H2O.traj", logfile="H2O.log")
optimizer.run(fmax=0.01)
energy_h2o_ev = h2o.get_potential_energy()

### Check the molecular references

Inspect the optimized structure and the energy shown above. Which change in geometry or energy matters for the chemical question in this section?

## Calculate adsorption energies

First, let us build the clean Pt(111) slab and calculate its energy. Next, we will keep each slab fixed and relax its adsorbate with the same settings. Then we will combine those values with the molecular references. Each stage has its own cell, so we can inspect every structure and result.


In [ ]:
# Build the clean Pt(111) slab and calculate its energy.
metal = "Pt"
nx, ny, nz = 2, 2, 3
slab = fcc111(metal, (nx, ny, nz), a=3.9936, vacuum=10.0)
slab.pbc = [True, True, True]
slab.set_constraint(FixAtoms(mask=slab.get_tags() > 0))
slab.calc = calculator
energy_clean_slab_ev = slab.get_potential_energy()
print(f"Clean slab energy = {energy_clean_slab_ev:.2f} eV")

### Check the clean Pt surface


In [ ]:
# Reuse the same relaxation for the three adsorbates.
energy_o_slab_ev   = relax_adsorbate(slab, "O", position="ontop")
energy_oh_slab_ev  = relax_adsorbate(slab, "OH", position="ontop")
energy_ooh_slab_ev = relax_adsorbate(slab, "OOH", position="ontop")

### Check the three adsorbates


In [ ]:
# Apply the same molecular corrections without overwriting the electronic energies.
free_energy_h2o_ev = energy_h2o_ev + 0.56 - 0.67
free_energy_h2_ev  = energy_h2_ev + 0.27 - 0.41

# Form each CHE reference explicitly: O, OH, and OOH.
free_energy_o_ev   = (energy_o_slab_ev   - energy_clean_slab_ev - free_energy_h2o_ev + free_energy_h2_ev + 0.07)
free_energy_oh_ev  = (energy_oh_slab_ev  - energy_clean_slab_ev - free_energy_h2o_ev + 0.5 * free_energy_h2_ev)
free_energy_ooh_ev = (energy_ooh_slab_ev - energy_clean_slab_ev - 2 * free_energy_h2o_ev + 1.5 * free_energy_h2_ev + 0.09)
print(f"Pt: E_O={free_energy_o_ev:.2f}, E_OH={free_energy_oh_ev:.2f}, E_OOH={free_energy_ooh_ev:.2f}")

### Read the intermediate energies

These three values set the ORR free energy levels. Keep their reference convention fixed as we move from one reaction step to the next.


In [ ]:
display(pd.DataFrame({"intermediate": ["O*", "OH*", "OOH*"], "free_energy_eV": [free_energy_o_ev, free_energy_oh_ev, free_energy_ooh_ev]}))

## Draw the free energy diagram

The `plot_at_U` function traces the ORR path from $*+\mathrm{O_2}$ through OOH*, O*, and OH* to $*+\mathrm{H_2O}$. It draws the path at a chosen potential. We will inspect the equilibrium potential (1.23 V), the limiting potential, and 0 V. The potential gap gives the thermodynamic overpotential.

> **Pause and predict:** At 1.23 V, which Pt(111) step may still require energy?
>
> **What to look for:** Find the least favorable step from its energy change, then confirm it in the plot.

The slider applies CHE energy shifts to the stored intermediates.


In [ ]:
def plot_at_potential(potential_v, intermediate_energies_ev, color="#0000cc", metal="Pt"):
    states = [
        4.92 - 4 * potential_v,
        intermediate_energies_ev[0] - 3 * potential_v,
        intermediate_energies_ev[1] - 2 * potential_v,
        intermediate_energies_ev[2] - 1 * potential_v,
        0,
    ]
    plt.plot([-0.2, 0.2], [states[0], states[0]], c=f"{color}", linestyle="-", linewidth=4, label=f"{metal}, $U$={potential_v} V")
    plt.plot([0.8, 1.2], [states[1], states[1]], c=f"{color}", linestyle="-", linewidth=4)
    plt.plot([1.8, 2.2], [states[2], states[2]], c=f"{color}", linestyle="-", linewidth=4)
    plt.plot([2.8, 3.2], [states[3], states[3]], c=f"{color}", linestyle="-", linewidth=4)
    plt.plot([3.8, 4.2], [states[4], states[4]], c=f"{color}", linestyle="-", linewidth=4)
    plt.plot([0.2, 0.8], [states[0], states[1]], c=f"{color}", linestyle=":", linewidth=1)
    plt.plot([1.2, 1.8], [states[1], states[2]], c=f"{color}", linestyle=":", linewidth=1)
    plt.plot([2.2, 2.8], [states[2], states[3]], c=f"{color}", linestyle=":", linewidth=1)
    plt.plot([3.2, 3.8], [states[3], states[4]], c=f"{color}", linestyle=":", linewidth=1)

In [ ]:
def plot_free_energy_interactive(potential_v, reference_curves, energies_ev, catalyst):
    # A tuple of energies fixes this example while the potential slider changes U.
    available = [4.92 - energies_ev[0], energies_ev[0] - energies_ev[1], energies_ev[1] - energies_ev[2], energies_ev[2]]
    limiting_v = min(available)
    fig, ax = plt.subplots()
    if reference_curves:
        plot_at_potential(1.23, energies_ev, colors[1], catalyst)
        plot_at_potential(round(limiting_v, 2), energies_ev, colors[2], catalyst)
    plot_at_potential(potential_v, energies_ev, colors[0], catalyst)
    ax.set_xlabel("Reaction coordinate")
    ax.set_ylabel("Free energies / eV")
    ax.set_xticks([0, 1, 2, 3, 4])
    ax.set_xticklabels(["*+O$_2$", "OOH*", "O*", "OH*", "*+H$_2$O"])
    ax.set_ylim(-3, 6)
    ax.set_title(f"{catalyst}: U = {potential_v:.2f} V vs RHE")
    ax.legend()
    fig.tight_layout()
    plt.show()

In [ ]:
intermediate_energies_ev = [free_energy_ooh_ev, free_energy_o_ev, free_energy_oh_ev]
overpotential_v = round(1.23 - min(4.92 - intermediate_energies_ev[0], intermediate_energies_ev[0] - intermediate_energies_ev[1], intermediate_energies_ev[1] - intermediate_energies_ev[2], intermediate_energies_ev[2]), 2)

In [ ]:
# Capture these energies now so later examples do not change this widget.
potential_plot_1 = partial(plot_free_energy_interactive, energies_ev=tuple(intermediate_energies_ev), catalyst="Pt")
potential_controls_1 = {"potential_v": FloatSlider(value=0.0, min=-0.2, max=1.5, step=0.01, description="U / V vs RHE", continuous_update=False), "reference_curves": Checkbox(value=True, description="Reference curves"),}

In [ ]:
# Move U; optionally hide the fixed equilibrium and limiting potential curves.
potential_output_1 = interactive_output(potential_plot_1, potential_controls_1)
display(HBox([potential_output_1, VBox(list(potential_controls_1.values()))]))

## Compare with literature DFT data

So far, UMA has supplied our energies. Now we will read literature DFT data that use the BEEF-vdW functional. The values come from the [Zenodo adsorption-energy database](https://zenodo.org/records/19023993).

> **Pause and predict:** With the same reference convention, what pattern might these two energy sources share?
>
> **What to look for:** Keep a method change separate from a change in reference energies or corrections.


In [ ]:
# Download once into the current folder.
!if [ ! -s molecules.db ]; then wget -q https://zenodo.org/records/19023993/files/molecules.db -O molecules.db; fi
if not Path('molecules.db').is_file() or Path('molecules.db').stat().st_size == 0:
    raise FileNotFoundError("Download failed: molecules.db")
# Download once into the current folder.
!if [ ! -s slab_database.db ]; then wget -q https://zenodo.org/records/19023993/files/slab_database.db -O slab_database.db; fi
if not Path('slab_database.db').is_file() or Path('slab_database.db').stat().st_size == 0:
    raise FileNotFoundError("Download failed: slab_database.db")
# Download once into the current folder.
!if [ ! -s adsorbate_database.db ]; then wget -q https://zenodo.org/records/19023993/files/adsorbate_database.db -O adsorbate_database.db; fi
if not Path('adsorbate_database.db').is_file() or Path('adsorbate_database.db').stat().st_size == 0:
    raise FileNotFoundError("Download failed: adsorbate_database.db")

Let us read the bare slab table. We will keep rows that use the BEEF-vdW functional, then place them in a pandas DataFrame.


In [ ]:
adsorption_records = []
slab_database = connect("slab_database.db")
for row in slab_database.select(xc="BEEF-vdW"):
    adsorption_records.append({"metal": row.structure_str.split("_")[0], "adsorbate": "_", "position": "_", "energy": row.get("energy"),})
adsorption_table = pd.DataFrame(adsorption_records)
adsorption_table

Now let us prepare the adsorbate table. We will keep atop sites that have a reported energy.


In [ ]:
adsorbate_database = connect("adsorbate_database.db")
for row in adsorbate_database.select(xc="BEEF-vdW"):
    structure_parts = row.structure_str.split("_")
    position = structure_parts[3]
    energy = row.get("energy")
    if position == "top" and energy is not None:
        adsorption_records.append({"metal": structure_parts[0], "adsorbate": structure_parts[2], "position": position, "energy": energy,})
adsorption_table = pd.DataFrame(adsorption_records)
adsorption_table

The adsorption table is ready. Let us retrieve the BEEF-vdW molecular references for each metal. The loop selects a clean slab and applies the stated O, OH, or OOH reference equation. These are electronic adsorption energies; we will track thermochemical corrections as separate terms.


In [ ]:
molecule_database = connect("molecules.db")
energy_h2o_ev = molecule_database.get(xc="BEEF-vdW", formula="H2O").energy
energy_h2_ev = molecule_database.get(xc="BEEF-vdW", formula="H2").energy

In [ ]:
# Use each database row directly, with its matching clean metal slab.
adsorption_energy_values = []

In [ ]:
# .iterrows() visits a table one row at a time; row contains the named column values.
# .iloc[0] selects the first row by position; check that the selection is nonempty.
for index, row in adsorption_table.iterrows():
    adsorbate = row["adsorbate"]
    if adsorbate == "_":
        adsorption_energy_values.append(None)
        continue

    metal = row["metal"]
    clean_slab_row = adsorption_table[(adsorption_table["metal"] == metal) & (adsorption_table["adsorbate"] == "_")].iloc[0]
    energy_clean_slab_ev = clean_slab_row["energy"]
    # Use the first matching adsorbate entry in the table.
    adsorbate_row = adsorption_table[(adsorption_table["metal"] == metal) & (adsorption_table["adsorbate"] == adsorbate)].iloc[0]
    energy_adsorbate_slab_ev = adsorbate_row["energy"]

    if adsorbate == "OH":
        energy = energy_adsorbate_slab_ev - energy_clean_slab_ev - energy_h2o_ev + 0.5 * energy_h2_ev
    elif adsorbate == "O":
        energy = energy_adsorbate_slab_ev - energy_clean_slab_ev - energy_h2o_ev + energy_h2_ev
    elif adsorbate == "OOH":
        energy = energy_adsorbate_slab_ev - energy_clean_slab_ev - 2 * energy_h2o_ev + 1.5 * energy_h2_ev
    else:
        raise ValueError(f"Unknown adsorbate: {adsorbate}")
    adsorption_energy_values.append(energy)

In [ ]:
adsorption_table["adsorption_energy"] = adsorption_energy_values
df_filtered = adsorption_table.dropna()

In [ ]:
df_filtered

Let us apply the same free energy diagram to Au with the literature DFT data. Then we can compare it with the UMA result above.


In [ ]:
metal = "Pt"
intermediate_energies_ev = [
    adsorption_table[
        (adsorption_table["metal"] == metal) & (adsorption_table["adsorbate"] == "OOH")
    ]["adsorption_energy"].values[0],
    adsorption_table[
        (adsorption_table["metal"] == metal) & (adsorption_table["adsorbate"] == "O")
    ]["adsorption_energy"].values[0],
    adsorption_table[
        (adsorption_table["metal"] == metal) & (adsorption_table["adsorbate"] == "OH")
    ]["adsorption_energy"].values[0],
]
overpotential_v = round(1.23 - min((4.92 - intermediate_energies_ev[0], intermediate_energies_ev[0] - intermediate_energies_ev[1], intermediate_energies_ev[1] - intermediate_energies_ev[2], intermediate_energies_ev[2])), 2)

In [ ]:
# Capture these energies now so later examples do not change this widget.
potential_plot_2 = partial(plot_free_energy_interactive, energies_ev=tuple(intermediate_energies_ev), catalyst=metal)
potential_controls_2 = {"potential_v": FloatSlider(value=0.0, min=-0.2, max=1.5, step=0.01, description="U / V vs RHE", continuous_update=False), "reference_curves": Checkbox(value=True, description="Reference curves"),}

In [ ]:
# Move U; optionally hide the fixed equilibrium and limiting potential curves.
potential_output_2 = interactive_output(potential_plot_2, potential_controls_2)
display(HBox([potential_output_2, VBox(list(potential_controls_2.values()))]))

## Student exercise: Find the limiting potential

Use OOH, O, and OH free energies to list each of the four ORR step energies at zero potential. Find the limiting potential. Copy OH free energy into a new variable, change it by 0.10 eV, and repeat. Explain which step sets the limit. State why this calculation tests sensitivity in the model.

Record a prediction, one numerical or graphical result, and your chemical interpretation. Keep the stored values intact.


In [ ]:
# Your code here.
# Use a separate variable for your changed parameter and display the comparison.

### Your answer

Prediction: [write your prediction]

Result: [report a value or refer to your figure]

Chemical explanation: [explain the result]

Assumption or limitation: [state one limitation]


## What have you learned

Let us gather the main ideas from this notebook. You can now:

- Convert compatible adsorption energies into ORR intermediate free energies.
- Track electron counts and potential shifts along the reaction path.
- Find the potential limiting step and calculate a thermodynamic overpotential.
- Explain how the energy source and corrections shape the comparison.


### Questions for discussion

1. Why do ORR intermediate levels shift by different amounts as the potential changes?
2. A path may have no uphill free energy step and still proceed slowly. Which information could reveal the rate?
